In [7]:
import random
# Une solution = un chromosome
# Chaque géne = (cour_id, salle_di, créneau_id, jour_id)

# (cours, salle, créneau, jour)
# Générer 30 chromosomes aléatoires avec chaque valeur entre 0 et 5
chromosome = [(random.randint(0, 5), random.randint(0, 5), random.randint(0, 5), random.randint(0, 5)) for _ in range(30)]

print(chromosome)

# Fonction Fitness
def fintness_emploi_du_temps(chromosome):
    score = 1000 # Score maximum théorique
    
    # 1. Pénalité pour les conflits graves
    conflits = detecter_conflits(chromosome)
    score -= len(conflits) * 50 # -50 par conflit
    
    # 2. Salles trop petites
    for cours, salle, creneau, jour in chromosome:
        if capacite_salle[salle] < etudiants_par_cours[cours]:
            score -= 30
            
    # 3. Préférences des enseignants
    for prof in professeurs:
        emploi = emploi_du_prof(prof, chromosome)
        if emploi_trop_charge(emploi):
            score -= 20
        if repect_preferences(prof, emploi):
            score -= 10
            
    return max(score, 0) # Score minimum de 0

# Croisement et Mutation
def crossover_emploi_du_temps(parent1, parent2):
    point_croisement = random.randint(1, len(parent1) - 1)
    enfant = parent1[:point_croisement] + parent2[point_croisement:]
    return enfant

def mutuation_emploi_du_temps(chromosome, taux_mutation=0.01):
    for i in range(len(chromosome)):
        if random.random() < taux_mutation:
            cours, salle, creneau, jour = chromosome[i]
            
            # Mutation type 1: Changer la salle
            if random.random() < 0.5:
                nouvelle_salle = random.choice(salles_disponibles)
                chromosome[i] = (cours, nouvelle_salle, creneau, jour)
            
            # Mutation type 2: Changer le créneau
            else:
                nouveau_creneau = random.randint(0, 4)
                chromosome[i] = (cours, salle, nouveau_creneau, jour)
    return chromosome

# Exemple d'utilisation
def algorithme_genetique_emploi_du_temps():
    # 1. Initialisation de la population
    population = [generer_emploi_aleatoire() for _ in range(100)]
    
    meilleur_score = 0
    meilleur_solution = None
    derniers_socres = []  # Initialisation pour suivre les scores
    
    for generation in range(500): # Nombre de générations
        # 2. Évaluation de la population
        scores = [fintness_emploi_du_temps(ind) for ind in population]
        
        # 3. Sauvegarde de la meilleure solution
        max_score = max(scores)
        if max_score > meilleur_score:
            meilleur_score = max_score
            meilleur_solution = population[scores.index(max_score)]
        
        # Mise à jour de l'historique des scores
        derniers_socres.append(max_score)
        if len(derniers_socres) > 50:
            derniers_socres.pop(0)
            
        # 4. Condition d'arrét
        if generation > 50 and pas_damelioration(derniers_socres):
            break
        
        # 5. Sélection
        parents = selection_par_tournoi(population, scores, k=50)
        
        # 6. Reproduction
        enfants = []
        while len(enfants) < len(population):
            parent1, parent2 = random.sample(parents, 2)
            enfant = crossover_emploi_du_temps(parent1, parent2)
            enfant = mutuation_emploi_du_temps(enfant, taux_mutation=0.05)
            enfants.append(enfant)
            
        # 7. Nouvelle génération (élitisme : garde les 5% meilleurs)
        population = enfants
        # Recalculer les scores pour la nouvelle population avant d'appliquer l'élitisme
        scores_nouveaux = [fintness_emploi_du_temps(ind) for ind in population]
        garder_meilleurs(population, scores_nouveaux, top_percent=0.05)
        
    return meilleur_solution, meilleur_score

[(3, 5, 5, 5), (2, 3, 1, 1), (1, 2, 1, 5), (0, 1, 1, 4), (0, 2, 2, 1), (3, 2, 2, 2), (4, 5, 2, 3), (1, 2, 5, 2), (2, 2, 5, 5), (3, 0, 0, 3), (2, 3, 3, 5), (2, 2, 1, 3), (4, 0, 4, 2), (5, 3, 2, 1), (1, 3, 4, 2), (3, 3, 1, 5), (5, 5, 1, 3), (0, 0, 2, 2), (5, 5, 4, 2), (2, 4, 2, 3), (5, 3, 1, 3), (5, 5, 2, 0), (3, 5, 5, 0), (5, 0, 4, 1), (4, 1, 2, 4), (0, 5, 4, 1), (1, 1, 1, 3), (0, 1, 5, 0), (4, 0, 2, 5), (5, 5, 1, 5)]


In [8]:
# ===== DONNÉES DE CONFIGURATION =====
# Capacité des salles (salle_id -> capacité)
capacite_salle = {0: 20, 1: 30, 2: 40, 3: 50, 4: 60, 5: 100}

# Nombre d'étudiants par cours (cours_id -> nombre_etudiants)
etudiants_par_cours = {0: 25, 1: 30, 2: 35, 3: 40, 4: 45, 5: 50}

# Liste des professeurs (chaque prof a un ID et enseigne certains cours)
professeurs = [
    {'id': 0, 'cours': [0, 1], 'pref_creneaux': [0, 1, 2], 'max_heures': 4},
    {'id': 1, 'cours': [2, 3], 'pref_creneaux': [1, 2, 3], 'max_heures': 5},
    {'id': 2, 'cours': [4, 5], 'pref_creneaux': [2, 3, 4], 'max_heures': 4},
]

# Salles disponibles
salles_disponibles = list(capacite_salle.keys())

# ===== FONCTIONS MANQUANTES =====

def detecter_conflits(chromosome):
    """
    Détecte les conflits dans l'emploi du temps.
    Un conflit = même salle, même créneau, même jour pour des cours différents.
    """
    conflits = []
    for i in range(len(chromosome)):
        for j in range(i + 1, len(chromosome)):
            cours1, salle1, creneau1, jour1 = chromosome[i]
            cours2, salle2, creneau2, jour2 = chromosome[j]
            
            # Conflit si même salle, même créneau, même jour, mais cours différents
            if salle1 == salle2 and creneau1 == creneau2 and jour1 == jour2 and cours1 != cours2:
                conflits.append((i, j))
    return conflits

def emploi_du_prof(prof, chromosome):
    """
    Retourne l'emploi du temps d'un professeur (tous les cours qu'il enseigne).
    """
    emploi = []
    for cours, salle, creneau, jour in chromosome:
        if cours in prof['cours']:
            emploi.append((cours, salle, creneau, jour))
    return emploi

def emploi_trop_charge(emploi):
    """
    Vérifie si un emploi du temps est trop chargé (plus de 6 heures par jour).
    """
    heures_par_jour = {}
    for cours, salle, creneau, jour in emploi:
        if jour not in heures_par_jour:
            heures_par_jour[jour] = 0
        heures_par_jour[jour] += 1
    
    # Si un jour a plus de 6 heures, c'est trop chargé
    return any(heures > 6 for heures in heures_par_jour.values())

def repect_preferences(prof, emploi):
    """
    Vérifie si l'emploi du temps respecte les préférences du professeur.
    Retourne True si les préférences ne sont PAS respectées (pour pénaliser).
    """
    for cours, salle, creneau, jour in emploi:
        if creneau not in prof['pref_creneaux']:
            return True  # Préférence non respectée
    return False  # Toutes les préférences sont respectées

def generer_emploi_aleatoire():
    """
    Génère un emploi du temps aléatoire (30 cours).
    """
    return [(random.randint(0, 5), random.randint(0, 5), random.randint(0, 5), random.randint(0, 5)) for _ in range(30)]

def selection_par_tournoi(population, scores, k=50):
    """
    Sélectionne k parents par tournoi (on prend le meilleur de 3 individus aléatoires).
    """
    parents = []
    for _ in range(k):
        # Sélectionner 3 individus aléatoires
        indices = random.sample(range(len(population)), min(3, len(population)))
        # Prendre celui avec le meilleur score
        meilleur_idx = max(indices, key=lambda i: scores[i])
        parents.append(population[meilleur_idx])
    return parents

def garder_meilleurs(population, scores, top_percent=0.05):
    """
    Garde les meilleurs individus dans la population (élitisme).
    Modifie la population en place.
    """
    n_meilleurs = max(1, int(len(population) * top_percent))
    indices_tries = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    meilleurs_indices = indices_tries[:n_meilleurs]
    meilleurs = [population[i] for i in meilleurs_indices]
    # Remplacer les pires par les meilleurs
    pires_indices = indices_tries[-n_meilleurs:]
    for idx, meilleur_idx in enumerate(pires_indices):
        population[meilleur_idx] = meilleurs[idx % len(meilleurs)]

def pas_damelioration(derniers_socres, seuil=0.01):
    """
    Vérifie s'il n'y a pas eu d'amélioration significative récemment.
    """
    if len(derniers_socres) < 10:
        return False
    
    # Vérifier si le score a augmenté de moins de seuil% sur les dernières générations
    score_initial = derniers_socres[0]
    score_final = derniers_socres[-1]
    
    if score_initial == 0:
        return False
    
    amelioration = (score_final - score_initial) / score_initial
    return amelioration < seuil

# Alias pour la fonction mutation (le code utilise mutation_emploi_du_temps mais la fonction s'appelle mutuation_emploi_du_temps)
def mutation_emploi_du_temps(chromosome, taux_mutation=0.01):
    return mutuation_emploi_du_temps(chromosome, taux_mutation)

# ===== TEST ET AFFICHAGE =====
print("✓ Configuration chargée avec succès!")
print(f"  - {len(capacite_salle)} salles disponibles")
print(f"  - {len(professeurs)} professeurs")
print(f"  - {len(etudiants_par_cours)} cours différents")
print("\n✓ Toutes les fonctions sont définies et prêtes à être utilisées.")
print("\nPour exécuter l'algorithme génétique, utilisez la cellule suivante.")


✓ Configuration chargée avec succès!
  - 6 salles disponibles
  - 3 professeurs
  - 6 cours différents

✓ Toutes les fonctions sont définies et prêtes à être utilisées.

Pour exécuter l'algorithme génétique, utilisez la cellule suivante.


In [10]:
# ===== EXÉCUTION DE L'ALGORITHME GÉNÉTIQUE =====
print("Démarrage de l'algorithme génétique...")
print("Cela peut prendre quelques secondes...\n")

meilleur_solution, meilleur_score = algorithme_genetique_emploi_du_temps()

print("=" * 60)
print("RÉSULTATS")
print("=" * 60)
print(f"Meilleur score obtenu: {meilleur_score}")
print(f"Nombre de conflits dans la meilleure solution: {len(detecter_conflits(meilleur_solution))}")
print(f"\nAperçu de la meilleure solution (10 premiers cours):")
print("-" * 60)
for i, (cours, salle, creneau, jour) in enumerate(meilleur_solution[:10]):
    capacite = capacite_salle[salle]
    etudiants = etudiants_par_cours[cours]
    statut_salle = "[OK]" if capacite >= etudiants else "[ERROR] (trop petite)"
    print(f"Cours {cours:2d}: Salle {salle} (cap. {capacite:3d}), Créneau {creneau}, Jour {jour} | Étudiants: {etudiants:2d} {statut_salle}")

if len(meilleur_solution) > 10:
    print(f"\n... et {len(meilleur_solution) - 10} autres cours")


Démarrage de l'algorithme génétique...
Cela peut prendre quelques secondes...

RÉSULTATS
Meilleur score obtenu: 970
Nombre de conflits dans la meilleure solution: 0

Aperçu de la meilleure solution (10 premiers cours):
------------------------------------------------------------
Cours  2: Salle 3 (cap.  50), Créneau 5, Jour 4 | Étudiants: 35 [OK]
Cours  2: Salle 5 (cap. 100), Créneau 4, Jour 1 | Étudiants: 35 [OK]
Cours  0: Salle 1 (cap.  30), Créneau 4, Jour 5 | Étudiants: 25 [OK]
Cours  2: Salle 4 (cap.  60), Créneau 0, Jour 0 | Étudiants: 35 [OK]
Cours  1: Salle 2 (cap.  40), Créneau 4, Jour 5 | Étudiants: 30 [OK]
Cours  1: Salle 5 (cap. 100), Créneau 5, Jour 4 | Étudiants: 30 [OK]
Cours  1: Salle 4 (cap.  60), Créneau 1, Jour 1 | Étudiants: 30 [OK]
Cours  1: Salle 4 (cap.  60), Créneau 0, Jour 3 | Étudiants: 30 [OK]
Cours  2: Salle 2 (cap.  40), Créneau 4, Jour 1 | Étudiants: 35 [OK]
Cours  5: Salle 3 (cap.  50), Créneau 3, Jour 5 | Étudiants: 50 [OK]

... et 20 autres cours
